In [1]:
# Cell 1 — 讀取預測人流
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

DF_PRED_PATH = DATA_DIR / "predictions" / "convlstm_df_pred.parquet"

df_pred = pd.read_parquet(DF_PRED_PATH)

print("rows:", len(df_pred))
df_pred.head()

rows: 5808914


,d,t,x,y,score,count
0,32,18,1,22,0.941464,1
1,26,20,1,22,0.969777,1
2,31,20,1,22,0.982814,1
3,43,21,1,22,0.984589,1
4,52,21,1,22,0.976331,1


In [2]:
# Cell 2 — 計算每個 grid 的平均人流
grid_mean = (
    df_pred
    .groupby(["x","y"])["score"]
    .mean()
    .reset_index()
)

grid_mean.rename(columns={"score":"mean_crowd"}, inplace=True)

grid_mean.head()

,x,y,mean_crowd
0,1,22,0.973182
1,1,23,1.177460
2,1,24,1.084786
3,1,25,1.104661
4,1,34,1.473461


In [3]:
# Cell 3 — 自動分類城市區域
q1 = grid_mean.mean_crowd.quantile(0.25)
q2 = grid_mean.mean_crowd.quantile(0.50)
q3 = grid_mean.mean_crowd.quantile(0.75)

def classify_zone(c):
    
    if c > q3:
        return "city_center"
    
    elif c > q2:
        return "commercial"
    
    elif c > q1:
        return "residential"
    
    elif c > 0:
        return "suburb"
    
    else:
        return "mountain"

grid_mean["zone_type"] = grid_mean.mean_crowd.apply(classify_zone)

grid_mean.zone_type.value_counts()

zone_type
suburb         1521
city_center    1521
residential    1520
commercial     1520
Name: count, dtype: int64

In [4]:
# Cell 4 — 生成停車容量
rng = np.random.default_rng(42)

capacity_base = {
    "city_center": 30,
    "commercial": 60,
    "residential": 80,
    "suburb": 120,
    "mountain": 0
}

def gen_capacity(zone):

    base = capacity_base[zone]
    
    if base == 0:
        return 0
        
    noise = rng.normal(0, base*0.2)
    
    return max(0, int(base + noise))

grid_mean["parking_capacity"] = grid_mean.zone_type.apply(gen_capacity)

In [5]:
# Cell 5 — 空間停車需求比例
ratio_base = {
    "city_center": 0.25,
    "commercial": 0.40,
    "residential": 0.55,
    "suburb": 0.70,
    "mountain": 0.00
}

def gen_ratio(zone):

    base = ratio_base[zone]
    
    noise = rng.normal(0,0.05)
    
    return np.clip(base + noise,0,1)

grid_mean["parking_ratio_base"] = grid_mean.zone_type.apply(gen_ratio)

In [6]:
# Cell 6 — 加入「時間模型」
def time_multiplier(zone, t):

    hour = t * 0.5
    
    if zone in ["city_center","commercial"]:
        
        if 8 <= hour <= 18:
            return 1.4
        
        elif 18 < hour <= 23:
            return 0.7
        
        else:
            return 0.5

    elif zone in ["residential","suburb"]:
        
        if 18 <= hour <= 23:
            return 1.3
        
        elif 8 <= hour <= 18:
            return 0.8
        
        else:
            return 0.6

    else:
        return 0

In [7]:
# Cell 7 — 生成 grid + 時間停車需求
parking_time = []

for _, row in grid_mean.iterrows():
    
    x = row.x
    y = row.y
    zone = row.zone_type
    base_ratio = row.parking_ratio_base
    
    for t in range(48):
        
        mult = time_multiplier(zone,t)
        
        ratio = np.clip(base_ratio * mult,0,1)
        
        parking_time.append({
            "x":x,
            "y":y,
            "t":t,
            "parking_ratio":ratio
        })

parking_ratio_df = pd.DataFrame(parking_time)

In [8]:
# Cell 8 — 合併容量
parking_df = parking_ratio_df.merge(
    grid_mean[["x","y","parking_capacity","zone_type"]],
    on=["x","y"]
)

parking_df.head()


,x,y,t,parking_ratio,parking_capacity,zone_type
0,1,22,0,0.430864,127,suburb
1,1,22,1,0.430864,127,suburb
2,1,22,2,0.430864,127,suburb
3,1,22,3,0.430864,127,suburb
4,1,22,4,0.430864,127,suburb


In [12]:
# Cell 9 — 存檔 + Sanity Check
parking_df.to_parquet(
    "../data/parking/grid_parking_params.parquet",
    index=False
)

parking_df.groupby("zone_type").parking_ratio.mean()

zone_type
city_center    0.232390
commercial     0.372168
residential    0.463936
suburb         0.590734
Name: parking_ratio, dtype: float64